In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [0]:
source_data = [(1, 'A'), (2, 'B'), (3, 'C'), (4, 'D')]
source_columns = ['id','name']
target_data = [(1, 'A'), (2, 'B'), (4, 'X'), (5, 'F')]
target_columns = ['id','name']

source_df = spark.createDataFrame(data=source_data, schema=source_columns)
target_df = spark.createDataFrame(data=target_data, schema=target_columns)

In [0]:
merged_df = source_df.join(target_df, how = 'full', on ='id')
merged_df = merged_df.withColumn("source_name",source_df.name)\
            .withColumn("target_name",target_df.name)\
            .select("id","source_name","target_name")
merged_df.show()

In [0]:
df = (
    merged_df
    .withColumn(
        "comment",
        F.when(F.col("source_name").isNull(), "New in target")
         .when(F.col("target_name").isNull(), "New in source")
         .when(F.col("source_name") != F.col("target_name"), "misMatch")
    )
    .select("id", "comment")
    .filter(F.col("comment").isNotNull())
)

df.show()